# Analysis of metagenomic Nanopore sequencing data

We are going to look at data from sequencing 16S rRNA. In the real world, this type of 16S rRNA sequencing would often be done to identify the organisms in a sample of unknown composition, e.g. for diagnostic purposes. Here, we analyze data from a sample of a microbial community that was artificially compiled from eight different organisms: seven bacteria and one yeast.

## Submit your results for grading!

Work on the assignments and answer the questions in this Jupyter notebook. Submit your final notebook with code and produced output **on Moodle** (assignment of the course MDC_53661). Submission due **October 20th, 2026**.


Before submitting:
* run *Kernel → Restart & Run All* and check that all cells run without errors and show their output;
* name the file `MDC53661_<lastnames>_reads<N>.ipynb`, where `N` is the number of your reads file;
* `lastnames` should be the last names of people in your team, separated by '-'.
* upload only the notebook (no data files).

**Work in teams** Work in teams of two or three students! Only one of you should submit on Moodle! Don't forget to put your last names in the file name. Additionally define the list TEAM with your names below.

**Answer format.** Each assignment ends with an *answer cell*: a Python dictionary with fixed keys. Replace every `...` by your answer (numbers, lists, or short text), keep the keys unchanged, and run the cell. You can compute values with code (e.g. `"n_reads": len(data)`). Text answers should be short: at most the number of sentences given in the comment. Your code and plots above the answer cells remain part of the submission.

## Assignments

### Preassignment: Setup 

First, we need to install required bioinformatics software and packages in a conda environment.

From last time, you should have set up your computer appropriately:

* if you have Windows, you can run a PowerShell with WSL  
* you have installed miniforge / miniconda3
* you could run the notebook `computational-setup.ipynb` from the first session

In a terminal or PowerShell after executing wsl, create and activate a conda environment and install software e.g. as follows

```
conda create -n sequencing
conda activate sequencing
conda install jupyter biopython numpy pandas seaborn cd-hit mafft fasttree jalview
```

Note that you need to create the environment and install each package only once, but you need to activate the environment in every session.

Run `jupyter notebook` from the same terminal, where you activated the environment `sequencing`.

Open the url of the jupyter server in a browser or connect Visual Code to the existing server.

If you need more software, it should be installed in the same conda environment.

It should now work to import a series of Python modules as below. Try to get a brief idea of the imported Python packages.

In [16]:
# Biopython
import Bio
from Bio import SeqIO

# some Python standard lib
import random

# Reading compressed data
import gzip

# python modules for data analysis
import numpy as np

# plotting of data
import matplotlib.pyplot as plt
import seaborn as sns

# Answers are printed in a fixed format that is read for grading (do not modify)
import json
def submit(question, answers):
    print(f"#ANSWER {question} " + json.dumps(answers, default=str))

In [32]:
# List your names (first and last name) of people in your team, e.g. ["John Doe", "Jane Smith"]
TEAM = [
    "Maxence Barberet--Pinto",
    "Nihat Ahmadli",
    "Jihene Megdiche",
]
submit("TEAM", TEAM)

#ANSWER TEAM ["Maxence Barberet--Pinto", "Nihat Ahmadli", "Jihene Megdiche"]


### Assignment 1: Download sequencing data and first look

From the course page on Moodle, download one of the files reads_N.fastq.gz in the folder Reads.
Please don't all download the same file. It will be more interesting to compare results for different files. To choose for your team, e.g. generate a random number in [0..9]:

In [18]:
print(f"Choose reads number {random.randint(0, 9)}.")

Choose reads number 2.


This file is a gzip compressed file in fastq format. It contains reads from sequencing the 16S rDNA in a metagenomic sample. It also contains the quality of base calling for each single base; this information is encoded in "quality strings" (see next assignment).

Specifically, our sets of reads each contain 10% of data from the first hour of a Nanopore sequencing run (of 72h), which we performed in the course of a MODAL in September 2025. 

**Questions Q1**:
* Which file did you download? 
* How large is the file and how many reads are in the file?
* Which read lengths do you observe for the first few reads? Which lengths would you expect?

You can use the following shell commands and/or biopython code to help answering the questions.

In [33]:
myfile = "reads_2.fastq.gz"

In [45]:
%pwd
!ls

data_acquisition  miniforge3


In [36]:
## use ls to determine file size
!ls -l $myfile

## show the first 16 lines in the file (= the first 4 reads)
!gunzip -c $myfile | head -n16

## determine uncompressed file size by counting characters
!gunzip -c $myfile | wc -c

# count lines (attention: there are several lines per sequence)
!gunzip -c $myfile | wc -l

# count lines starting with '@'
# Careful: '@' also encodes quality Q31, so quality lines may start with '@' as well!
# Compare to the line count divided by 4. Which count is correct?
!gunzip -c $myfile | grep -c '^@'

ls: cannot access 'reads_2.fastq.gz': No such file or directory
gzip: reads_2.fastq.gz: No such file or directory
gzip: reads_2.fastq.gz: No such file or directory
0
gzip: reads_2.fastq.gz: No such file or directory
0
gzip: reads_2.fastq.gz: No such file or directory
0


In [ ]:
# YOUR ANSWERS TO Q1: replace each ... and run the cell
Q1 = {
    "reads_file": "reads_2.fastq.gz",        # name of your file, e.g. "reads_3.fastq.gz"
    "first_read_id": "3179d30e-6525-4455-96f3-2f5357a7956f",     # id of the first read: header line after '@' up to the first space
    "file_size_bytes": 31196717,   # size of the compressed file in bytes (ls -l)
    "n_reads": 27012,           # number of reads in the file
    "first_lengths": first_lengths,     # list with the lengths of the first 4 reads
    "expected_length": "We expect reads of about 1500-1600 nt, because the full-length 16S rRNA gene is about 1500 bp, plus the primers and Nanopore adapters/barcodes added during library preparation. Shorter reads like the 4th one (623 nt) can come from fragmented DNA or truncated sequencing.",   # text (max. 2 sentences): which read lengths do you expect, and why?
}
submit("Q1", Q1)

In [ ]:
# YOUR ANSWERS TO Q1: replace each ... and run the cell
Q1 = {
    "reads_file": "reads_2.fastq.gz",        # name of your file, e.g. "reads_3.fastq.gz"
    "first_read_id": "3179d30e-6525-4455-96f3-2f5357a7956f",     # id of the first read: header line after '@' up to the first space
    "file_size_bytes": 31196717,   # size of the compressed file in bytes (ls -l)
    "n_reads": 27012,           # number of reads in the file
    "first_lengths": [1604, 1608, 1626, 623],     # list with the lengths of the first 4 reads
    "expected_length": "We expect reads of about 1500-1600 nt, because the full-length 16S rRNA gene is about 1500 bp, plus the primers and Nanopore adapters/barcodes added during library preparation. Shorter reads like the 4th one (623 nt) can come from fragmented DNA or truncated sequencing.",   # text (max. 2 sentences): which read lengths do you expect, and why?
}
submit("Q1", Q1)

### Assignment 2: Reading sequencing data in Python

Here you are going to read in the fastq.gz file using Biopython to better understand its contents.


**Questions Q2**: 
* How many sequences are in the file?
* What is the minimal, mean, and maximal length of sequences in the file?
* What are the minimal and maximal base calling qualities (Phred scores) in the file?
* What is the mean Phred quality over all bases in the file? Also report the mean error probability per base (convert each Q to $P=10^{-Q/10}$ before averaging). Why do the two kinds of averages differ?

You are supposed to **write and submit Python code** to answer Questions Q2.

***HINTS***: The excursion below explains quality scores; you will also find some useful example code.

#### Excursion: fastq files and how to read fastq.gz

Fastq files specify sequences and position-wise base calling quality as Phred Q-Scores

$\displaystyle Q=-10\ \log _{10}P,$

where $P$ is the probability that the base is called incorrectly.

 
Quality scores are stored encoded as quality strings.

In Python, we can read the files using SeqIO, which translates quality score codes to numbers. The meaning of codes is specified as follows (but we don't need to take care ourselves).
```
qscores_codes = "!\"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~"
```

Have a look at the raw contents of the file, e.g. using the shell commands of Assignment 1 to display the first few lines of the fastq.gz file.

##### Example code: read fastq.gz

In [22]:
def read_fasta_gz(filename,format="fasta"):
    data = []
    with gzip.open(filename, "rt") as handle: 
        for record in SeqIO.parse(handle, format):
            data.append(record)
    return data

def read_fastq_gz(filename):
    return read_fasta_gz(filename,"fastq")


From each fastq.gz we can obtain vectors sequence records, which give us access to sequences and qualities.

For example, after reading the file contents into data, like `data = read_fastq_gz("filename.fastq.gz")`,
we get for each sequence record in data, 
 
 * the sequence as `record.seq` and 
 
 * the quality as `record.letter_annotations["phred_quality"]`

In [23]:
# YOUR CODE TO ANSWER Q2

In [24]:
# YOUR ANSWERS TO Q2: replace each ... and run the cell
Q2 = {
    "n_reads": ...,
    "length_min": ...,
    "length_mean": ...,
    "length_max": ...,
    "phred_min": ...,          # over all bases in the file
    "phred_max": ...,
    "phred_mean": ...,         # mean of the Phred scores of all bases
    "error_prob_mean": ...,    # mean of P = 10^(-Q/10) over all bases
    "why_averages_differ": ..., # text (max. 3 sentences)
}
submit("Q2", Q2)

#ANSWER Q2 {"n_reads": "Ellipsis", "length_min": "Ellipsis", "length_mean": "Ellipsis", "length_max": "Ellipsis", "phred_min": "Ellipsis", "phred_max": "Ellipsis", "phred_mean": "Ellipsis", "error_prob_mean": "Ellipsis", "why_averages_differ": "Ellipsis"}


### Assignment 3: Filter the data by length

Write code to filter the data by sequence length. Keep only reads with length between 1400 and 1700nt. Write the results to a uncompressed fasta file.

***Question Q3:*** 
* How many reads pass the filter? 
* Why should it make sense to filter in this way?
* How could we explain much shorter and much longer reads in our data?

##### Example code: convert fastq.gz to fasta

In [25]:
def convert_fastq_gz(infile, outfile):
    with gzip.open(infile, "rt") as infh:
        with open(outfile, "wt") as outfh: 
            for record in SeqIO.parse(infh, "fastq"):
                SeqIO.write(record,outfh,"fasta")

In [26]:
# YOUR CODE

#### Assignment 3.1: Plot a histogram of the lengths in the file

One can generate the plot using seaborn, like `sns.histplot(lengths)`, after preparing `lengths` appropriately.


In [27]:
# YOUR CODE

In [28]:
# YOUR ANSWERS TO Q3: replace each ... and run the cell
Q3 = {
    "n_filtered": ...,         # number of reads with length between 1400 and 1700 nt
    "why_filter": ...,         # text (max. 3 sentences)
    "short_long_reads": ...,   # text (max. 3 sentences): how to explain much shorter / longer reads?
}
submit("Q3", Q3)

#ANSWER Q3 {"n_filtered": "Ellipsis", "why_filter": "Ellipsis", "short_long_reads": "Ellipsis"}


### Assignment 4: Cluster the sequences with CD-hit

Cluster the filtered reads using cd-hit. Then, answer the following questions (more explanations below):

***Questions Q4:***

* How are the cluster sizes distributed? (draw a histogram)
* How large are your largest clusters and how many clusters have size 1?
* How does the clustering change for a different threshold in cd-hit, at 97% identity?

*Runtime:* clustering at 95% takes a few minutes; at 97% it takes about ten times longer (start it early; use `-T 0` to use all CPU cores).


Here is a command that clusters at a threshold of 95% identity:
```cd-hit-est -i input.fasta -o clusters -c 0.95 -n 8 -d 0 -M 0 -T 0```

Run ```cd-hit-est``` to see a description of its parameters.

In [29]:
!cd-hit-est

/bin/bash: line 1: cd-hit-est: command not found


The command writes a fasta file `clusters` of cluster representatives as well as a clustering report `clusters.clstr`. The latter file can be parsed to obtain a
list of lists of the sequence ids in each cluster.

Here is example code to read the cluster report file:

In [30]:
def read_clstr(clstr_file):
    clusters = list()
    with open(clstr_file) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">Cluster"):
                clusters.append(list())
            else:
                seq_id = line.split(">")[1].split("...")[0]
                clusters[-1].append(seq_id)

    return clusters

clusters = read_clstr("clusters.clstr")

print(len(clusters), "clusters; the first one:", clusters[0])

FileNotFoundError: [Errno 2] No such file or directory: 'clusters.clstr'

In [ ]:
# YOUR CODE

In [ ]:
# YOUR ANSWERS TO Q4: replace each ... and run the cell
Q4 = {
    "n_clusters_95": ...,      # number of clusters at 95% identity
    "largest5_95": ...,        # list: sizes of the 5 largest clusters at 95%
    "n_singletons_95": ...,    # number of clusters of size 1 at 95%
    "n_clusters_97": ...,      # same at 97% identity
    "largest5_97": ...,
    "n_singletons_97": ...,
    "comparison": ...,         # text (max. 3 sentences): how and why does the clustering change?
}
submit("Q4", Q4)

### Assignment 5: Multiple alignment and phylogenetic tree of the largest cluster

Generate a multiple alignment and phylogenetic tree of the largest cluster (details below). Then, briefly discuss the following questions:

***Questions Q5***
* How much time does it take to align and to build the tree? (Use `%%time` at the top of a cell to measure.)
* How do you interpret the tree? Can you identify subclusters?

Optionally look at the next largest clusters in the same way...

#### Details for Assignment 5
Start by generating a fasta file of all sequences in the largest cluster. From `read_clstr`, we can obtain a list of sequence ids of the sequences in the largest cluster. Here is example code to go through the input file of the clustering and collect all entries in the largest cluster. Here, `largest_cluster_ids` is meant to be a `set` of sequence ids that you need to define yourself (membership tests in sets are fast, in lists they are slow):

In [ ]:
with open("reads_filtered.fasta", "rt") as infh:
    with open("largest_cluster.fasta", "wt") as outfh: 
        for record in SeqIO.parse(infh, "fasta"):
            seq_id=record.id
            if seq_id in largest_cluster_ids:
                SeqIO.write(record,outfh,"fasta")

Next, use the alignment tool mafft to align the sequences and write the result to a fasta file. View the multiple alignment using `jalview` (or using some server online).

In [ ]:
!mafft --thread 8 --auto largest_cluster.fasta  > largest_cluster_aligned.fasta

Finally use FastTree to produce a phylogenetic tree from the alignment.
The results are written in NEWICK format, a string representation of a tree, to the file largest_cluster_tree.nwk.

View the phylogenetic tree online at https://itol.embl.de/upload.cgi
The server allows you to upload the produced newick file.

FastTree can be used in the following way (call `FastTree` without arguments to learn about the possible arguments):

In [ ]:
!FastTree -noml -nt largest_cluster_aligned.fasta > largest_cluster_tree.nwk

In [ ]:
# YOUR ANSWERS TO Q5: replace each ... and run the cell
Q5 = {
    "n_sequences": ...,        # number of sequences in the largest cluster (at 95%)
    "time_align_s": ...,       # wall time of mafft in seconds
    "time_tree_s": ...,        # wall time of FastTree in seconds
    "tree_interpretation": ..., # text (max. 4 sentences): how do you interpret the tree? subclusters?
}
submit("Q5", Q5)

### Assignment 6: Identify some organisms

Choose reads from your largest cluster(s), e.g. the representative sequence and some other ones, and find known similar (homologous) sequences using BLAST on the NCBI Blast server:

https://blast.ncbi.nlm.nih.gov/Blast.cgi?PROGRAM=blastn&PAGE_TYPE=BlastSearch&LINK_LOC=blasthome


CAREFUL: Select the rRNA/ITS databases and choose
16S ribosomal RNA sequences (Bacteria and Archaea) from the menu.

*Alternatively*, you can run BLAST locally on your computer against the same NCBI 16S database (see the optional section *Local BLAST* below).

***Questions Q6***:
* How can the phylogenetic tree of the cluster help to select candidate sequences for BLAST?
* Which organisms do you identify according to the best BLAST hits?
* How clearly can the organism be identified? Compare scores and E-values!


Optionally, identify some more organisms from either the best cluster or the next largest ones.

#### Optional: Local BLAST

Instead of the web server, you can install BLAST in your conda environment, download NCBI's 16S rRNA database once, and run `blastn` from the notebook.

| | NCBI web server | local BLAST |
|---|---|---|
| setup | none | `conda install blast` + one-time database download |
| many queries | one by one, by hand (limits on NCBI usage) | many reads in one run (e.g. all cluster representatives) |
| reproducibility | results live on a web page; database is updated without notice | command, parameters and database version are in your notebook |
| output | graphical, with links to taxonomy and records | text/table; you parse it in Python |
| speed | queue at NCBI | fast for a small database like 16S |

Note: both default to `megablast` (word size 28) for nucleotide queries; use `-task blastn` for more sensitive searches.

Download and extract the database (once):

In [ ]:
RUN_LOCAL_BLAST = False   # set to True if you use local BLAST

if RUN_LOCAL_BLAST:
    import os
    os.makedirs("blastdb", exist_ok=True)
    # update_blastdb.pl comes with BLAST from conda (conda install blast)
    !cd blastdb && update_blastdb.pl --decompress 16S_ribosomal_RNA

Then BLAST a fasta file of query sequences (e.g. the cluster representatives written by cd-hit to `clusters`, or a few reads of the largest cluster). The tabular output format (`-outfmt 6`) is easy to read with pandas; `stitle` adds the description of the hit:

In [ ]:
import pandas as pd

if RUN_LOCAL_BLAST:
    !blastn -query queries.fasta -db blastdb/16S_ribosomal_RNA -max_target_seqs 5 -outfmt "6 qseqid sacc stitle pident length evalue bitscore" -out blast_hits.tsv

    hits = pd.read_csv("blast_hits.tsv", sep="\t", header=None,
                       names=["query", "accession", "title", "pident", "length", "evalue", "bitscore"])
    hits.head(20)

In [ ]:
# YOUR ANSWERS TO Q6: replace each ... and run the cell
# One entry per read you BLASTed (add or remove entries as needed); report the best hit.
Q6 = {
    "hits": [
        {"read_id": ..., "cluster_size": ..., "species": ..., "identity_percent": ..., "evalue": ...},
        {"read_id": ..., "cluster_size": ..., "species": ..., "identity_percent": ..., "evalue": ...},
    ],
    "tree_helps": ...,         # text (max. 3 sentences): how can the tree help to select reads for BLAST?
    "how_clear": ...,          # text (max. 3 sentences): how clearly are organisms identified (scores, E-values)?
}
submit("Q6", Q6)